# 추천 방식 3개 - 시나리오별 결과 비교

같은 요청을 세 방식에 그대로 넣고, 각각 무엇을 추천하는지와 **어떤 조건을 어겼는지**를 본다.

| | 방식 |
|---|---|
| A | 합성 고객을 군집화하고, 입력이 속한 세그먼트가 쓰는 요금제 |
| B | 요금제를 스펙으로 군집화하고, 입력에 가까운 군집에서 싼 순 |
| C | 필터 + 랭킹 (`recommend()`) |

정답 요금제가 없으므로 "어느 게 정확한가"는 재지 않는다. **어디서 무너지는가**를 본다.

합성 고객이 없으면 먼저 `python src/make_synthetic.py`를 한 번 돌린다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd() / "src"))

from compare_methods import FilterMethod, PlanClusterMethod, SegmentMethod
from fair_price import attach_value_score
from make_synthetic import OUT_PATH as SYNTHETIC_PATH

# 회귀 학습이 몇 초 걸린다. 한 번만 돌리고 아래 셀에서 재사용한다.
plans = attach_value_score()
customers = pd.read_csv(SYNTHETIC_PATH, encoding="utf-8-sig")
methods = [SegmentMethod(customers), PlanClusterMethod(plans), FilterMethod()]

print(f"요금제 {len(plans):,}개 · 합성 고객 {len(customers):,}명 · 방식 {len(methods)}개")

## 시나리오

여기를 고쳐서 원하는 조건으로 돌려 본다. 키는 `recommend()` 인자와 같다.

- `data_unlimited=True`면 `data_gb`는 무시된다
- `mvno_ok=False`면 통신 3사만

In [ ]:
SCENARIOS = {
    "대학생 · 2만원 · 20GB · 알뜰폰 OK": dict(
        budget=20_000, data_gb=20.0, data_unlimited=False,
        mvno_ok=True, voice_unlimited=False, age=25),
    "직장인 · 5만원 · 50GB · 3사만 · 통화무제한": dict(
        budget=50_000, data_gb=50.0, data_unlimited=False,
        mvno_ok=False, voice_unlimited=True, age=25),
    "영상 헤비 · 8만원 · 무제한 · 통화무제한": dict(
        budget=80_000, data_gb=None, data_unlimited=True,
        mvno_ok=True, voice_unlimited=True, age=25),
    "절약형 · 1만원 · 5GB · 알뜰폰 OK": dict(
        budget=10_000, data_gb=5.0, data_unlimited=False,
        mvno_ok=True, voice_unlimited=False, age=25),
    "시니어 · 3만원 · 5GB · 3사만 · 통화무제한 · 65세": dict(
        budget=30_000, data_gb=5.0, data_unlimited=False,
        mvno_ok=False, voice_unlimited=True, age=65),
}

TOP = 3   # 시나리오·방식마다 몇 개까지 볼지

## 실행

`recommend.broken_rules()`가 결과 한 줄이 요청의 어느 조건을 어겼는지 돌려준다.
**필터의 거울**이라 제품 코드에 있고, `recommend.check()`가 같은 함수로 검증한다.
C(필터+랭킹)는 정의상 여기가 항상 비어야 한다 - 비지 않으면 `recommend()`가 깨진 것이다.

In [ ]:
from recommend import broken_rules


def show(label: str, ask: dict) -> None:
    print("=" * 84)
    print(f"## {label}")
    for method in methods:
        got = method(plans, **ask)
        print(f"\n  [{method.name}] {len(got)}건")
        if got.empty:
            print("    (없음)")
            continue
        for i in range(min(TOP, len(got))):
            row = got.iloc[i]
            bad = broken_rules(got.iloc[[i]], ask)   # 한 줄만 넣으면 그 요금제의 위반
            fee = pd.to_numeric(row["discounted_fee"], errors="coerce")
            data = "무제한" if row["data_unlimited"] is True else (
                f"{row['data_gb']:.0f}GB" if pd.notna(row["data_gb"]) else "?")
            print("    {:<38} {:>10} {:>7}  {:<5}{}".format(
                str(row["plan_name"])[:38],
                f"{int(fee):,}원" if pd.notna(fee) else "?",
                data, row["carrier_type"],
                "  << " + ", ".join(bad) if bad else ""))


for label, ask in SCENARIOS.items():
    show(label, ask)

## 군집형은 요청이 달라도 같은 답을 내는가

위반율 숫자보다 이쪽이 더 선명하다. 군집이 입력을 "가까운 덩어리"로 뭉개면,
예산이 5배 차이 나는 두 사람이 같은 추천을 받는다.

In [ ]:
for method in methods:
    groups: dict[tuple, list[str]] = {}
    for label, ask in SCENARIOS.items():
        groups.setdefault(tuple(method(plans, **ask)["plan_id"]), []).append(label)

    print(f"{method.name}: 서로 다른 결과 {len(groups)}가지 / 시나리오 {len(SCENARIOS)}개")
    for shared in groups.values():
        if len(shared) > 1:
            print("   같은 답:", " == ".join(s.split(" · ")[0] for s in shared))

## 전체 격자 200칸 요약

시나리오 5개는 손으로 고른 것이라 유리한 칸만 봤을 수 있다.
`compare_methods.py`의 격자 200칸을 그대로 돌려 위반율을 다시 확인한다. (수십 초)

In [ ]:
from compare_methods import evaluate, grid, jaccard, summarize

res = evaluate(methods, plans, grid())
display(summarize(res))
print("\n세 방식 결과 겹침 - Jaccard 평균")
display(jaccard(res))

---
# 부록 · A(세그먼트 분류) 해부

A가 왜 예산을 48.6% 어기는지는 요약 숫자만 봐서는 안 보인다. 세 토막으로 나눠서 본다.

| 단계 | 무엇을 | 어디에 |
|---|---|---|
| 1. 데이터 | 합성 고객 4만 명 (`data/synthetic/customers.csv`) | `src/make_synthetic.py` |
| 2. 모델 | `StandardScaler` → `KMeans(k=6)` | `src/compare_methods.py` `SegmentMethod` |
| 3. 예측 | 입력을 같은 축으로 세우고 `predict` → 그 세그먼트가 **실제로 쓰는** 요금제 상위 5개 | 같은 클래스 `__call__` |

**요금제를 고르는 게 아니라 사람을 분류한다.** 그래서 예산·데이터가 필터로 걸리지 않는다.

## 1. 합성 고객은 어떻게 만들어졌나

요금제를 가입자 수 비율로 뽑고, **그 요금제의 스펙에서 사람을 역산**한다.
`베이직70GB`가 뽑히면 → 데이터를 63~70GB 쓰는 사람, 예산 = 그 요금제 값.

순환이 있다(요금제 → 사람 → 요금제). 분포만큼은 가입자 수 316만 건 실측에서 온다.

In [ ]:
print(f"합성 고객 {len(customers):,}명, 원본이 된 요금제 {customers['source_plan_id'].nunique()}종\n")
display(customers.head(5))
display(customers[["age", "data_gb_month", "budget_krw"]].describe().round(1))

print("데이터 무제한 희망 :", f"{customers['data_unlimited_need'].mean():.1%}",
      "  (이 사람들은 data_gb_month가 비어 있다)")
print("통화 무제한 희망  :", f"{customers['voice_unlimited_need'].mean():.1%}",
      "  <- 뒤에서 이 한 축이 거리를 지배한다")
print("알뜰폰 비중      :", f"{customers['carrier_type'].eq('MVNO').mean():.1%}",
      "  (조사값 4.1%에 맞춘 것. 요금제 수 기준으로는 MVNO가 80%)")

## 2. 모델 — 무엇을 축으로 삼았나

축은 4개(`SEG_FEATURES`)뿐이다. `StandardScaler`가 각 축을 `(값 − 평균) / 표준편차`로 바꾼 뒤
`KMeans`가 6덩어리로 나눈다.

**표준편차를 눈여겨본다.** 표준편차가 작은 축은 조금만 달라도 거리가 크게 벌어진다.

In [ ]:
from compare_methods import SEG_FEATURES, UNLIMITED_GB, _vec

A = methods[0]                                    # SegmentMethod
scaler = A.model.named_steps["standardscaler"]
kmeans = A.model.named_steps["kmeans"]

print("축:", SEG_FEATURES, "| 무제한은", UNLIMITED_GB, "GB로 치환\n")
display(pd.DataFrame({"평균": scaler.mean_, "표준편차": scaler.scale_},
                     index=SEG_FEATURES).round(2))

print("군집 중심 (원래 단위로 되돌림)")
display(pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                     columns=SEG_FEATURES).round(1))

In [ ]:
# 세그먼트 프로필 + 그 세그먼트가 실제로 쓰는 요금제 상위 5개
tagged = customers.assign(seg=A.labels)
display(tagged.groupby("seg").agg(
    인원=("customer_id", "size"),
    예산중앙=("budget_krw", "median"),
    데이터중앙=("data_gb_month", "median"),
    무제한비율=("data_unlimited_need", "mean"),
    통화무제한비율=("voice_unlimited_need", "mean"),
    나이중앙=("age", "median"),
).round(2))

catalog = plans.drop_duplicates("plan_id").set_index("plan_id")
for seg, ids in A.plans_by_segment.items():
    print(f"\n[seg {seg}] 이 세그먼트가 쓰는 요금제")
    for pid in ids:
        if pid not in catalog.index:
            print("    (요금제 테이블에 없음)", pid)
            continue
        r = catalog.loc[pid]
        print("    {:<40} {:>10} {}".format(
            str(r["plan_name"])[:40], f"{int(r['discounted_fee']):,}원", r["carrier_type"]))

## 3. 직접 찍어보기

`explain()`에 조건을 넣으면 **입력이 세그먼트로 바뀌는 과정 전부**를 펼쳐 준다.

- 입력 벡터 → 표준화 값(평균에서 몇 표준편차인지) → 6개 중심까지의 거리 → 배정된 세그먼트
- 그 세그먼트가 내놓는 추천과, 그게 요청의 어느 조건을 어겼는지

In [ ]:
def explain(*, budget, data_gb=None, data_unlimited=False,
            voice_unlimited=False, age=None, mvno_ok=True):
    """입력 하나가 어느 세그먼트로, 왜 그리로 가는지 펼쳐 본다."""
    ask = dict(budget=budget, data_gb=data_gb, data_unlimited=data_unlimited,
               voice_unlimited=voice_unlimited, age=age, mvno_ok=mvno_ok)
    vec = _vec(SEG_FEATURES, **ask)
    z = scaler.transform(vec)[0]
    dist = kmeans.transform(scaler.transform(vec))[0]
    seg = int(A.model.predict(vec)[0])

    print("입력 (원래 단위 / 표준화 값)")
    for name, raw, std in zip(SEG_FEATURES, vec.to_numpy()[0], z):
        flag = "   <- 평균에서 크게 벗어남" if abs(std) > 2 else ""
        print(f"    {name:<16} {raw:>10.1f}   {std:>+6.2f} 표준편차{flag}")

    print("\n각 세그먼트 중심까지 거리 (가까운 곳으로 배정)")
    for i, d in enumerate(dist):
        print(f"    seg {i}  {d:6.2f}" + ("   <- 배정" if i == seg else ""))

    got = A(plans, **ask)
    print(f"\nseg {seg} 가 내놓는 추천 {len(got)}건")
    for i in range(len(got)):
        row = got.iloc[i]
        bad = broken_rules(got.iloc[[i]], ask)
        print("    {:<40} {:>10}  {}".format(
            str(row["plan_name"])[:40],
            f"{int(row['discounted_fee']):,}원" if pd.notna(row["discounted_fee"]) else "?",
            "<< " + ", ".join(bad) if bad else "충족"))
    return got


_ = explain(budget=20_000, data_gb=20.0, age=25)                                # 대학생
print("\n" + "=" * 76 + "\n")
_ = explain(budget=80_000, data_unlimited=True, voice_unlimited=True, age=25)   # 영상 헤비

## 4. 축을 하나씩 흔들어 본다

어느 축이 세그먼트를 실제로 가르는지 보는 가장 빠른 방법이다. 나머지를 고정하고 하나만 훑는다.

In [ ]:
def sweep(axis: str, values: list, **fixed) -> None:
    held = {k: v for k, v in fixed.items() if k != axis}
    print(f"[{axis}] 만 바꾼다   고정: {held}")
    for v in values:
        seg = int(A.model.predict(_vec(SEG_FEATURES, **{**fixed, axis: v}))[0])
        print(f"    {axis}={str(v):>8}  ->  seg {seg}")
    print()


base = dict(budget=20_000, data_gb=20.0, data_unlimited=False,
            voice_unlimited=False, age=25)

sweep("budget", [5_000, 10_000, 30_000, 50_000, 80_000, 150_000], **base)
sweep("age", [20, 30, 40, 50, 60, 70, 80], **base)
sweep("voice_unlimited", [False, True], **base)

# 통화 무제한을 켠 뒤 다시 예산을 훑으면 그제서야 세그먼트가 갈린다.
sweep("budget", [5_000, 10_000, 30_000, 50_000, 80_000, 150_000],
      **{**base, "voice_unlimited": True})